In [2]:
import pandas as pd
from scipy.sparse import load_npz
from tqdm import tqdm
import numpy as np
import os
import gc
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

tqdm.pandas()

# Leitura dos dados

In [3]:
embeddings = pd.read_parquet("../../data/datasets/embeddings.parquet")

In [33]:
embedding_cols = ['embedding__codefuse_ai__F2LLM_v2_14B__texto',
       'embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado',
       'embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado_sem_justificativa',
       'embedding__Octen__Octen_Embedding_8B__texto',
       'embedding__Octen__Octen_Embedding_8B__texto_preprocessado',
       'embedding__Octen__Octen_Embedding_8B__texto_preprocessado_sem_justificativa',
       'embedding__Qwen__Qwen3_Embedding_8B__texto',
       'embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado',
       'embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado_sem_justificativa',
       'embedding__nvidia__llama_embed_nemotron_8b__texto',
       'embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado',
       'embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado_sem_justificativa',
       'embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto',
       'embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto_preprocessado',
       'embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto_preprocessado_sem_justificativa',
       'embedding__joaorobson__harrier_oss_v1_27b__texto',
       'embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado',
       'embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado_sem_justificativa',
       'embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto',
       'embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto_preprocessado',
       'embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto_preprocessado_sem_justificativa',
       'embedding__openai__text_embedding_3_large__texto',
       'embedding__openai__text_embedding_3_large__texto_preprocessado',
       'embedding__openai__text_embedding_3_large__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__trunc128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__trunc128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__trunc128__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto_preprocessado_sem_justificativa',
       'embedding__josedossantos__bertimbau_tuned__trunc512__texto',
       'embedding__josedossantos__bertimbau_tuned__trunc512__texto_preprocessado',
       'embedding__josedossantos__bertimbau_tuned__trunc512__texto_preprocessado_sem_justificativa',
       'embedding__josedossantos__bertimbau_tuned__meanpool512__texto',
       'embedding__josedossantos__bertimbau_tuned__meanpool512__texto_preprocessado',
       'embedding__josedossantos__bertimbau_tuned__meanpool512__texto_preprocessado_sem_justificativa',
       'embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto',
       'embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado',
       'embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado_sem_justificativa',
       'embedding__gemini__gemini_embedding_2__raw__texto',
       'embedding__gemini__gemini_embedding_2__raw__texto_preprocessado',
       'embedding__gemini__gemini_embedding_2__raw__texto_preprocessado_sem_justificativa',
       'embedding__gemini__gemini_embedding_2__sts__texto',
       'embedding__gemini__gemini_embedding_2__sts__texto_preprocessado',
       'embedding__gemini__gemini_embedding_2__sts__texto_preprocessado_sem_justificativa'
]

In [76]:
embeddings.groupby(["materia", "tema"]).size().groupby("materia").agg(
    media="mean",
    mediana="median",
    desvio="std",
    min="min",
    max="max"
)

,media,mediana,desvio,min,max
materia,,,,,
MPV_612_2013,3.793103,1.0,9.818991,1,60
PEC_6_2019,9.241379,8.0,7.337064,1,28
PLP_68_2024,7.563218,3.0,11.313064,1,85


In [35]:
embeddings.query("materia == 'PLP_68_2024'") \
    .groupby("tema_nivel_2") \
    .size() \
    .agg(
        media="mean",
        mediana="median",
        desvio="std",
        min="min",
        max="max",
        quantidade="count"
    )

media          19.352941
mediana         7.000000
desvio         25.537643
min             1.000000
max           118.000000
quantidade    102.000000
dtype: float64

In [36]:
embeddings.query("materia == 'PLP_68_2024'") \
    .groupby("tema_macro") \
    .size() \
    .agg(
        media="mean",
        mediana="median",
        desvio="std",
        min="min",
        max="max",
        quantidade="count"
    )

media          56.400000
mediana        17.000000
desvio        112.426499
min             1.000000
max           475.000000
quantidade     35.000000
dtype: float64

In [37]:
(embeddings["texto_preprocessado"]
.str.split()
.str.len()
.groupby(embeddings["materia"])
.agg(["mean", "std"]))

,mean,std
materia,,
MPV_612_2013,345.781818,381.845229
PEC_6_2019,499.541045,467.714975
PLP_68_2024,619.674265,547.459315


In [38]:
print("Mean and std. of embeddings values")
for col in embedding_cols:

    normas = embeddings[col].apply(
        lambda x: np.linalg.norm(np.array(x))
    )

    print(
        f"{col:90s} "
        f"mean={normas.mean():.4f} "
        f"std={normas.std():.4f}"
    )

Mean and std. of embeddings values
embedding__codefuse_ai__F2LLM_v2_14B__texto                                                mean=1.0001 std=0.0020
embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado                                  mean=1.0000 std=0.0020
embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado_sem_justificativa                mean=1.0000 std=0.0020
embedding__Octen__Octen_Embedding_8B__texto                                                mean=1.0001 std=0.0013
embedding__Octen__Octen_Embedding_8B__texto_preprocessado                                  mean=1.0002 std=0.0013
embedding__Octen__Octen_Embedding_8B__texto_preprocessado_sem_justificativa                mean=1.0001 std=0.0013
embedding__Qwen__Qwen3_Embedding_8B__texto                                                 mean=1.0002 std=0.0022
embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado                                   mean=1.0001 std=0.0022
embedding__Qwen__Qwen3_Embedding_8B__texto_preprocess

## Avaliação do retrieval

* MAP
* R-Precision
* Precision@1
* MRR

In [39]:
def average_precision_from_hits(hits):
    n_rel = hits.sum()
    if n_rel == 0:
        return np.nan

    precisions = []
    hit_count = 0
    for i, rel in enumerate(hits, start=1):
        if rel:
            hit_count += 1
            precisions.append(hit_count / i)
            
    return np.mean(precisions)


def retrieval_metrics_from_scores(score_matrix, labels):
    labels = np.asarray(labels)
    n_docs = len(labels)
    
    # Ordena os índices do maior score para o menor
    ranking = np.argsort(-score_matrix, axis=1)
    
    p1_scores = []
    mrr_scores = []
    rprec_scores = []
    ap_scores = []
    
    for i in range(n_docs):
        label = labels[i]
        
        # Cria a máscara do Ground Truth desconsiderando o próprio documento 'i'
        relevant_mask = (labels == label)
        relevant_mask[i] = False
        R = relevant_mask.sum()
        
        if R == 0:
            continue
            
        # Filtra o ranking para remover o auto-hit (documento 'i')
        ranked = ranking[i]
        ranked = ranked[ranked != i]
        
        # Mapeia as labels ordenadas preditas
        predicted_labels = labels[ranked]
        
        # Hits booleanos (1 se relevante, 0 caso contrário)
        hits = (predicted_labels == label).astype(int)
        
        # 1. Precision@1
        p1_scores.append(hits[0])
        
        # 2. MRR (Reciprocal Rank do 1º resultado relevante)
        # np.where encontra as posições dos acertos; se houver acerto, pegamos o 1º (+1 pois indexação começa em 0)
        first_hit_indices = np.where(hits == 1)[0]
        if len(first_hit_indices) > 0:
            reciprocal_rank = 1.0 / (first_hit_indices[0] + 1)
        else:
            reciprocal_rank = 0.0
        mrr_scores.append(reciprocal_rank)
        
        # 3. R-Precision (Corte na quantidade exata de relevantes R)
        rprec_scores.append(np.mean(hits[:R]))
        
        # 4. Average Precision (MAP)
        ap = average_precision_from_hits(hits)
        if not np.isnan(ap):
            ap_scores.append(ap)
            
    return {
        "precision@1": float(np.mean(p1_scores)) if p1_scores else 0.0,
        "mrr": float(np.mean(mrr_scores)) if mrr_scores else 0.0,
        "r_precision": float(np.mean(rprec_scores)) if rprec_scores else 0.0,
        "map": float(np.mean(ap_scores)) if ap_scores else 0.0,
    }


def retrieval_metrics_embeddings(X, labels):
    # Calcula similaridade angular via cosseno
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, -np.inf)
    
    return retrieval_metrics_from_scores(sim, labels)



In [40]:
### ndcg
def compute_dcg(relevances, k=None):
    relevances = np.asarray(relevances)
    if k is not None:
        relevances = relevances[:k]
    if len(relevances) == 0:
        return 0.0
    gains = 2.0**relevances - 1.0
    discounts = np.log2(np.arange(1, len(relevances) + 1) + 1.0)
    return np.sum(gains / discounts)


def compute_ndcg(actual_relevances, k=None):
    actual_dcg = compute_dcg(actual_relevances, k=k)
    ideal_relevances = np.sort(actual_relevances)[::-1]
    ideal_dcg = compute_dcg(ideal_relevances, k=k)
    if ideal_dcg == 0.0:
        return 0.0
    return actual_dcg / ideal_dcg

def hierarchical_ndcg_from_scores(
    score_matrix,
    labels_original,
    labels_intermediate,
    labels_general,
    top_k_ndcg=(5, 10, None),
):
    labels_orig = np.asarray(labels_original)
    labels_inter = np.asarray(labels_intermediate)
    labels_gen = np.asarray(labels_general)

    n_docs = len(labels_orig)
    ranking = np.argsort(-score_matrix, axis=1)
    ndcg_scores = {f"ndcg@{k}" if k else "ndcg_all": [] for k in top_k_ndcg}

    for i in range(n_docs):
        ranked = ranking[i]
        ranked = ranked[ranked != i]

        orig_match = labels_orig[ranked] == labels_orig[i]
        inter_match = labels_inter[ranked] == labels_inter[i]
        gen_match = labels_gen[ranked] == labels_gen[i]

        # Ganhos graduados: 3 (Original), 2 (Intermediário), 1 (Geral), 0 (Fora)
        relevances = np.zeros(len(ranked), dtype=int)
        relevances[gen_match] = 1
        relevances[inter_match] = 2
        relevances[orig_match] = 3

        if np.sum(relevances) == 0:
            continue

        for k in top_k_ndcg:
            metric_name = f"ndcg@{k}" if k else "ndcg_all"
            score = compute_ndcg(relevances, k=k)
            ndcg_scores[metric_name].append(score)

    return {k: float(np.mean(v)) if v else 0.0 for k, v in ndcg_scores.items()}

In [41]:
materias = ["PEC_6_2019", "MPV_612_2013", "PLP_68_2024"]

campos_textuais = [
    "texto",
    "texto_preprocessado",
    "texto_preprocessado_sem_justificativa",
]

resultados = {}
resultados_df = {}

for materia in materias:
  print(f"\n{'='*60}\nProcessando Proposição: {materia}\n{'='*60}")

  embeddings_materia = (
      embeddings[embeddings["materia"] == materia].copy().reset_index(drop=True)
  )

  niveis = ["tema"]
  if materia == "PLP_68_2024":
    niveis = ["tema_macro", "tema_nivel_2", "tema"]

  resultados[materia] = []

  # -------------------------------------------------
  # Matrizes de Similaridade BM25L
  # -------------------------------------------------
  print("Carregando matrizes BM25L pré-calculadas (.npy)...")
  bm25_models = {}
  for campo in campos_textuais:
    for tipo in ["raw", "preprocess"]:
      path_file = f"../../data/bm25l/doc_doc_scores/sim_matrix_{materia}_{campo}_{tipo}.npy"
      if os.path.exists(path_file):
        bm25_models[f"{campo}__{tipo}"] = np.load(path_file)
      else:
        print(f"  [Aviso] Arquivo não encontrado: {path_file}")

  # -------------------------------------------------
  # Execução das Métricas por Nível de Granularidade
  # -------------------------------------------------
  for nivel in niveis:
    labels = embeddings_materia[nivel].values
    print(
        f"\nAvaliando Granularidade: [Nível: {nivel}] | Amostras:"
        f" {len(labels)}"
    )

    # Identifica se é o nível original do PLP para injetar o NDCG multinível
    is_plp_original = (materia == "PLP_68_2024") and (nivel == "tema")
    if is_plp_original:
      labels_orig = embeddings_materia["tema"].values
      labels_inter = embeddings_materia["tema_nivel_2"].values
      labels_gen = embeddings_materia["tema_macro"].values

    # Evaluator 1: Embeddings Densos das LLMs
    for col in tqdm(embedding_cols, desc="Avaliando Vetores Densos"):
      X = np.vstack(embeddings_materia[col].values)
      sim = cosine_similarity(X)
      np.fill_diagonal(sim, -np.inf)

      metrics = retrieval_metrics_from_scores(sim, labels)

      # Calcula e anexa o NDCG hierárquico na mesma linha do tema original do PLP
      if is_plp_original:
        ndcg_metrics = hierarchical_ndcg_from_scores(
            sim, labels_orig, labels_inter, labels_gen
        )
        metrics.update(ndcg_metrics)

      resultados[materia].append(
          {"nivel": nivel, "embedding": col, **metrics}
      )

    # Evaluator 2: Matrizes Léxicas do BM25L
    for nome, score_matrix in bm25_models.items():
      metrics = retrieval_metrics_from_scores(score_matrix, labels)

      # Calcula e anexa o NDCG hierárquico para o BM25L
      if is_plp_original:
        ndcg_metrics = hierarchical_ndcg_from_scores(
            score_matrix, labels_orig, labels_inter, labels_gen
        )
        metrics.update(ndcg_metrics)

      resultados[materia].append({
          "nivel": nivel,
          "embedding": f"retrieval__bm25l__{nome}",
          **metrics,
      })

  # Consolidação e ordenação do ranking da matéria
  resultados_df[materia] = (
      pd.DataFrame(resultados[materia])
      .sort_values(["map", "r_precision", "precision@1"], ascending=False)
      .reset_index(drop=True)
  )

  # Exibe o Top 20 imediato na tela
  print(f"\n>>> TOP 20 RANKING - {materia} <<<")
  display(resultados_df[materia].head(20))

  # Limpeza de memória da iteração
  del bm25_models
  gc.collect()


Processando Proposição: PEC_6_2019
Carregando matrizes BM25L pré-calculadas (.npy)...

Avaliando Granularidade: [Nível: tema] | Amostras: 268


Avaliando Vetores Densos: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 51/51 [00:01<00:00, 27.30it/s]



>>> TOP 20 RANKING - PEC_6_2019 <<<


,nivel,embedding,precision@1,mrr,r_precision,map
0,tema,embedding__Octen__Octen_Embedding_8B__texto,0.867424,0.895425,0.669929,0.716369
1,tema,embedding__Octen__Octen_Embedding_8B__texto_pr...,0.867424,0.897080,0.658753,0.713185
2,tema,embedding__joaorobson__harrier_oss_v1_27b__tex...,0.856061,0.894553,0.653239,0.699050
3,tema,embedding__Qwen__Qwen3_Embedding_8B__texto_pre...,0.852273,0.893137,0.645340,0.694852
4,tema,embedding__Qwen__Qwen3_Embedding_8B__texto,0.856061,0.893263,0.640187,0.692511
5,tema,embedding__joaorobson__harrier_oss_v1_27b__texto,0.844697,0.888855,0.652145,0.690927
6,tema,embedding__codefuse_ai__F2LLM_v2_14B__texto_pr...,0.825758,0.875471,0.639876,0.684038
7,tema,embedding__gemini__gemini_embedding_2__sts__te...,0.852273,0.893453,0.621883,0.670707
8,tema,embedding__jinaai__jina_embeddings_v5_text_sma...,0.856061,0.896026,0.590560,0.642256
9,tema,embedding__jinaai__jina_embeddings_v5_text_sma...,0.795455,0.853279,0.588230,0.630028



Processando Proposição: MPV_612_2013
Carregando matrizes BM25L pré-calculadas (.npy)...

Avaliando Granularidade: [Nível: tema] | Amostras: 220


...

Avaliando Vetores Densos: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 51/51 [00:46<00:00,  1.10it/s]



Avaliando Granularidade: [Nível: tema] | Amostras: 1974


Avaliando Vetores Densos: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 51/51 [01:29<00:00,  1.76s/it]



>>> TOP 20 RANKING - PLP_68_2024 <<<


,nivel,embedding,precision@1,mrr,r_precision,map,ndcg@5,ndcg@10,ndcg_all
0,tema,embedding__gemini__gemini_embedding_2__sts__te...,0.837295,0.884099,0.588335,0.622326,0.819759,0.777055,0.838078
1,tema,embedding__Octen__Octen_Embedding_8B__texto_pr...,0.817750,0.871135,0.570928,0.606218,0.806219,0.759711,0.827055
2,tema,embedding__joaorobson__harrier_oss_v1_27b__texto,0.809826,0.864308,0.567635,0.605676,0.796069,0.751833,0.831745
3,tema,embedding__joaorobson__harrier_oss_v1_27b__tex...,0.811410,0.864705,0.568953,0.604211,0.795918,0.753002,0.830274
4,tema,embedding__Octen__Octen_Embedding_8B__texto,0.813524,0.868455,0.569140,0.603232,0.805433,0.759313,0.825699
5,tema,embedding__codefuse_ai__F2LLM_v2_14B__texto_pr...,0.824089,0.874946,0.565604,0.598048,0.810432,0.762822,0.825740
6,tema,embedding__Qwen__Qwen3_Embedding_8B__texto,0.807713,0.863557,0.556748,0.589823,0.797275,0.749314,0.819452
7,tema,embedding__Qwen__Qwen3_Embedding_8B__texto_pre...,0.816165,0.867719,0.554392,0.587796,0.797323,0.746346,0.819439
8,tema,embedding__gemini__gemini_embedding_2__sts__texto,0.784469,0.851404,0.535112,0.566203,0.775732,0.729888,0.813293
9,tema_nivel_2,embedding__gemini__gemini_embedding_2__sts__te...,0.916367,0.940286,0.529894,0.564309,NaN,NaN,NaN


In [45]:
def extract_text_type(name):
    if "texto_preprocessado_sem_justificativa" in name:
        return "texto_preprocessado_sem_justificativa"
    
    elif "texto_preprocessado" in name:
        return "texto_preprocessado"
    
    elif "texto" in name:
        return "texto"
    
    else:
        return "outro"

In [57]:
MODEL_SIZE = {
    "F2LLM_v2_14B": 14.0,
    "Octen_Embedding_8B": 8.0,
    "Qwen3_Embedding_8B": 8.0,
    "llama_embed_nemotron_8b": 8.0,
    "harrier_oss_v1_27b": 27.0,
    "KaLM_Embedding_Gemma3_12B_2511": 12.0,
    "jina_embeddings_v5_text_small": 0.6,      # 330M
    "serafim_335m_portuguese_pt_sentence_encoder": 0.335,
    "serafim_335m_portuguese_pt_sentence_encoder_ir": 0.335,
    "bertimbau_tuned": 0.335,                   # BERTimbau Large ≈335M
    "ICT_TIME_and_Querit_embedding_v1": 4
}

MODEL_FAMILY = {
    "F2LLM_v2_14B": "multilingual",
    "Octen_Embedding_8B": "multilingual",
    "Qwen3_Embedding_8B": "multilingual",
    "llama_embed_nemotron_8b": "multilingual",
    "harrier_oss_v1_27b": "multilingual",
    "KaLM_Embedding_Gemma3_12B_2511": "multilingual",
    "jina_embeddings_v5_text_small": "multilingual",      # 330M
    "serafim_335m_portuguese_pt_sentence_encoder": "specialized",
    "serafim_335m_portuguese_pt_sentence_encoder_ir": "specialized",
    "bertimbau_tuned": "specialized",                   # BERTimbau Large ≈335M
    "ICT_TIME_and_Querit_embedding_v1": "multilingual",
    "bm25l": "bm25",
    "gemini": "proprietary",
    "openai": "proprietary"
}


def get_model_size(name):
    for model, size in MODEL_SIZE.items():
        if model in name:
            return size
    return np.nan

def get_model_family(name):
    for model, size in MODEL_FAMILY.items():
        if model in name:
            return size
    return np.nan

In [99]:
df_all = pd.concat(
    [df.assign(materia=m) for m, df in resultados_df.items()],
    ignore_index=True
)
df_all["tipo_texto"] = df_all["embedding"].apply(extract_text_type)
df_all["familia"] = df_all.embedding.apply(get_model_family)

In [101]:
df_all.to_parquet("../../data/datasets/resultados_retrieval.parquet", index=False)

In [100]:
df_all

,nivel,embedding,precision@1,mrr,r_precision,map,materia,ndcg@5,ndcg@10,ndcg_all,tipo_texto,familia
0,tema,embedding__Octen__Octen_Embedding_8B__texto,0.867424,0.895425,0.669929,0.716369,PEC_6_2019,NaN,NaN,NaN,texto,multilingual
1,tema,embedding__Octen__Octen_Embedding_8B__texto_pr...,0.867424,0.897080,0.658753,0.713185,PEC_6_2019,NaN,NaN,NaN,texto_preprocessado,multilingual
2,tema,embedding__joaorobson__harrier_oss_v1_27b__tex...,0.856061,0.894553,0.653239,0.699050,PEC_6_2019,NaN,NaN,NaN,texto_preprocessado,multilingual
3,tema,embedding__Qwen__Qwen3_Embedding_8B__texto_pre...,0.852273,0.893137,0.645340,0.694852,PEC_6_2019,NaN,NaN,NaN,texto_preprocessado,multilingual
4,tema,embedding__Qwen__Qwen3_Embedding_8B__texto,0.856061,0.893263,0.640187,0.692511,PEC_6_2019,NaN,NaN,NaN,texto,multilingual
...,...,...,...,...,...,...,...,...,...,...,...,...
280,tema_nivel_2,embedding__josedossantos__bertimbau_tuned__mea...,0.741406,0.788396,0.239505,0.238021,PLP_68_2024,NaN,NaN,NaN,texto_preprocessado_sem_justificativa,specialized
281,tema,embedding__PORTULAN__serafim_335m_portuguese_p...,0.511886,0.608913,0.240404,0.232702,PLP_68_2024,0.496227,0.441459,0.668565,texto,specialized
282,tema_nivel_2,embedding__PORTULAN__serafim_335m_portuguese_p...,0.662391,0.738744,0.223799,0.211263,PLP_68_2024,NaN,NaN,NaN,texto,specialized
283,tema,embedding__PORTULAN__serafim_335m_portuguese_p...,0.486001,0.570829,0.206915,0.203111,PLP_68_2024,0.460896,0.404061,0.650428,texto,specialized


In [78]:
import pandas as pd
import numpy as np

# 1. Filtrar a base
df_filtered = df_all[
    (df_all["nivel"] == "tema") & 
    (df_all["tipo_texto"] == "texto_preprocessado")
].copy()

# 2. Mapeamento dos nomes brutos para os nomes exatos do LaTeX
def format_model_name(name):
    if "bm25l" in name:
        if "preprocess" in name.split("__")[-1]:
            return "BM25L (w/ pre-proces.)"
        return "BM25L"
    if "F2LLM_v2_14B" in name:
        return "F2LLM-v2-14B"
    if "harrier_oss_v1_27b" in name:
        return "harrier-oss-v1-27b"
    if "ICT_TIME_and_Querit" in name:
        return "ICT-TIME-and-Querit"
    if "jina_embeddings_v5_text_small" in name:
        return "jina-v5-text-small (tm)"
    if "KaLM_Embedding_Gemma3_12B" in name:
        return "KaLM-Gemma3-12B"
    if "llama_embed_nemotron_8b" in name:
        return "nemotron-8B"
    if "Octen_Embedding_8B" in name:
        return "Octen-Embedding-8B"
    if "Qwen3_Embedding_8B" in name:
        return "Qwen3-Embedding-8B"
    if "gemini_embedding_2" in name:
        if "sts" in name or "similarity" in name:
            return "gemini-embedding-2 (ss)"
        return "gemini-embedding-2 (none)"
    if "text_embedding_3_large" in name:
        return "text-embedding-3-large"
    if "bertimbau_tuned" in name:
        if "mean" in name or "mp" in name:
            return "BERTimbau-FT (mp)"
        return "BERTimbau-FT (trunc)"
    if "serafim" in name:
        is_ir = "ir" in name.lower()
        strategy = "mp" if ("mean" in name or "mp" in name) else "trunc"
        prefix = "Serafim IR" if is_ir else "Serafim Base"
        return f"{prefix} ({strategy})"
    return name

df_filtered["Model"] = df_filtered["embedding"].apply(format_model_name)

# 3. Criar a Tabela Pivotada
# Métricas a extrair
metrics = ["precision@1", "r_precision", "map", "ndcg_all"]

df_pivot = df_filtered.pivot_table(
    index="Model",
    columns="materia",
    values=metrics,
    aggfunc="first"
)

# 4. Estruturar a ordem exata das colunas do LaTeX
# (MPV: P@1, R-P, MAP) | (PEC: P@1, R-P, MAP) | (PLP: P@1, R-P, MAP, NDGC@all)
cols_order = [
    ("precision@1", "MPV_612_2013"),
    ("r_precision", "MPV_612_2013"),
    ("map", "MPV_612_2013"),
    ("precision@1", "PEC_6_2019"),
    ("r_precision", "PEC_6_2019"),
    ("map", "PEC_6_2019"),
    ("precision@1", "PLP_68_2024"),
    ("r_precision", "PLP_68_2024"),
    ("map", "PLP_68_2024"),
    ("ndcg_all", "PLP_68_2024"),
]

# Garantir que todas as colunas existem (preenche NaN se faltar)
for col in cols_order:
    if col not in df_pivot.columns:
        df_pivot[col] = np.nan

df_pivot = df_pivot[cols_order]

# Renomear colunas para cabeçalho duplo limpo
df_pivot.columns = pd.MultiIndex.from_tuples([
    ("MPV", "P@1"), ("MPV", "R-P"), ("MPV", "MAP"),
    ("PEC", "P@1"), ("PEC", "R-P"), ("PEC", "MAP"),
    ("PLP", "P@1"), ("PLP", "R-P"), ("PLP", "MAP"), ("PLP", "NDGC@all")
])

# 5. Definir a ordem exata das linhas conforme o artigo
model_order = [
    # Lexical Baselines
    "BM25L",
    "BM25L (w/ pre-proces.)",
    # Open-Weight Multilingual
    "F2LLM-v2-14B",
    "harrier-oss-v1-27b",
    "ICT-TIME-and-Querit",
    "jina-v5-text-small (tm)",
    "jina-v5-text-small (cls)",
    "KaLM-Gemma3-12B",
    "nemotron-8B",
    "Octen-Embedding-8B",
    "Qwen3-Embedding-8B",
    # Proprietary APIs
    "gemini-embedding-2 (none)",
    "gemini-embedding-2 (ss)",
    "text-embedding-3-large",
    # Domain / Specialized
    "BERTimbau-FT (mp)",
    "BERTimbau-FT (trunc)",
    "Serafim Base (mp)",
    "Serafim Base (trunc)",
    "Serafim IR (mp)",
    "Serafim IR (trunc)"
]

# Reindexar preservando apenas os que existem
existing_models = [m for m in model_order if m in df_pivot.index]
df_pivot = df_pivot.reindex(existing_models)

# 6. Exibição Formatada no Notebook (3 casas decimais)
display(df_pivot.style.format("{:.3f}", na_rep="--"))

In [31]:
df_all[(df_all.nivel == "tema") & (df_all.materia == "PLP_68_2024") & (df_all.tipo_texto =="texto_preprocessado")]

,nivel,embedding,precision@1,mrr,r_precision,map,materia,ndcg@5,ndcg@10,ndcg_all,tipo_texto
126,tema,embedding__gemini__gemini_embedding_2__sts__te...,0.837295,0.884099,0.588335,0.622326,PLP_68_2024,0.819759,0.777055,0.838078,texto_preprocessado
127,tema,embedding__Octen__Octen_Embedding_8B__texto_pr...,0.817750,0.871135,0.570928,0.606218,PLP_68_2024,0.806219,0.759711,0.827055,texto_preprocessado
129,tema,embedding__joaorobson__harrier_oss_v1_27b__tex...,0.811410,0.864705,0.568953,0.604211,PLP_68_2024,0.795918,0.753002,0.830274,texto_preprocessado
131,tema,embedding__codefuse_ai__F2LLM_v2_14B__texto_pr...,0.824089,0.874946,0.565604,0.598048,PLP_68_2024,0.810432,0.762822,0.825740,texto_preprocessado
133,tema,embedding__Qwen__Qwen3_Embedding_8B__texto_pre...,0.816165,0.867719,0.554392,0.587796,PLP_68_2024,0.797323,0.746346,0.819439,texto_preprocessado
136,tema,embedding__ICT_TIME_and_Querit__ICT_TIME_and_Q...,0.817750,0.867598,0.534817,0.561174,PLP_68_2024,0.789481,0.736407,0.811964,texto_preprocessado
137,tema,retrieval__bm25l__texto_preprocessado__preprocess,0.793450,0.850869,0.528529,0.558958,PLP_68_2024,0.771229,0.728665,0.809167,texto_preprocessado
139,tema,embedding__nvidia__llama_embed_nemotron_8b__te...,0.808769,0.861080,0.528156,0.557408,PLP_68_2024,0.781938,0.726938,0.802832,texto_preprocessado
140,tema,embedding__jinaai__jina_embeddings_v5_text_sma...,0.808769,0.862247,0.529498,0.556340,PLP_68_2024,0.780361,0.726204,0.797878,texto_preprocessado
144,tema,embedding__openai__text_embedding_3_large__tex...,0.802958,0.857330,0.520260,0.548547,PLP_68_2024,0.777158,0.724805,0.806177,texto_preprocessado


In [19]:
metricas = ["precision@1", "r_precision", "map", "mrr"]


In [20]:
df_media_texto = df_all[df_all.nivel == 'tema'].groupby(["materia","nivel", "tipo_texto"])[metricas].mean()

In [55]:
df_media_texto

precision@1  \
materia      nivel tipo_texto                                           
MPV_612_2013 tema  texto                                     0.687683   
                   texto_preprocessado                       0.741421   
                   texto_preprocessado_sem_justificativa     0.702580   
PEC_6_2019   tema  texto                                     0.686147   
                   texto_preprocessado                       0.769661   
                   texto_preprocessado_sem_justificativa     0.691739   
PLP_68_2024  tema  texto                                     0.723291   
                   texto_preprocessado                       0.775967   
                   texto_preprocessado_sem_justificativa     0.758232   

                                                          r_precision  \
materia      nivel tipo_texto                                           
MPV_612_2013 tema  texto                                     0.471271   
                   texto_preprocessado                       0.527174   
                   texto_preprocessado_sem_justificativa     0.450292   
PEC_6_2019   tema  texto                                     0.441875   
                   texto_preprocessado                       0.510697   
                   texto_preprocessado_sem_justificativa     0.370632   
PLP_68_2024  tema  texto                                     0.438528   
                   texto_preprocessado                       0.478104   
                   texto_preprocessado_sem_justificativa     0.422629   

                                                               map       mrr  
materia      nivel tipo_texto                                                 
MPV_612_2013 tema  texto                                  0.503294  0.774361  
                   texto_preprocessado                    0.562412  0.818147  
                   texto_preprocessado_sem_justificativa  0.486326  0.786867  
PEC_6_2019   tema  texto                                  0.480448  0.759451  
                   texto_preprocessado                    0.552612  0.826392  
                   texto_preprocessado_sem_justificativa  0.401176  0.755210  
PLP_68_2024  tema  texto                                  0.458251  0.794330  
                   texto_preprocessado                    0.500525  0.833551  
                   texto_preprocessado_sem_justificativa  0.436602  0.813458

## Melhores modelos no geral

In [44]:
import pandas as pd

metricas = ["precision@1", "r_precision", "map", "mrr"]

ranks = []

for metrica in metricas:
    r = (
        df_all[(df_all.tipo_texto == "texto_preprocessado") & (df_all.nivel == "tema")].groupby("materia")
          .apply(lambda x:
                 x.assign(
                     rank=x[metrica].rank(
                         ascending=False,
                         method="average"
                     )
                 )[["embedding", "materia", "rank"]]
          )
          .reset_index(drop=True)
    )
    r["metrica"] = metrica
    ranks.append(r)

ranks = pd.concat(ranks)

ranking_medio = (
    ranks.groupby("embedding")
         .agg(
             ranking_medio=("rank","mean"),
             desvio=("rank","std"),
             melhor=("rank","min"),
             pior=("rank","max")
         )
         .sort_values("ranking_medio")
)

ranking_medio

C:\Users\3675-Robson\AppData\Local\Temp\ipykernel_9476\654025301.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x:
C:\Users\3675-Robson\AppData\Local\Temp\ipykernel_9476\654025301.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x:
C:\Users\3675-Robson\AppData\Local\Temp\ipykernel_9476\654025301.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This b

,ranking_medio,desvio,melhor,pior
embedding,,,,
embedding__Octen__Octen_Embedding_8B__texto_preprocessado,1.916667,1.892969,1.0,7.5
embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado,2.625000,1.150593,2.0,5.0
embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado,4.125000,1.350505,3.0,7.5
embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado,5.083333,3.604501,1.0,13.0
embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado,5.625000,1.625437,2.5,8.0
embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto_preprocessado,6.708333,2.840121,2.0,10.0
embedding__openai__text_embedding_3_large__trunc8192__texto_preprocessado,7.125000,1.208399,5.0,9.0
embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto_preprocessado,7.333333,3.984820,1.0,11.0
embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado,7.500000,1.430194,4.5,9.0


In [84]:
df_exp = df_all[
    (df_all["tipo_texto"] == "texto_preprocessado") & 
    (df_all["nivel"] == "tema")
].copy()

# Lista de métricas a ranquear (retrieval e clustering)
metricas = ["precision@1", "r_precision", "map"]

# Filtra apenas as métricas presentes nas colunas
metricas_validas = [m for m in metricas if m in df_exp.columns]

# =====================================================================
# 2. TRANSFORMAÇÃO PARA FORMATO LONGO E CÁLCULO DOS RANKS
# =====================================================================
# Derrete o DataFrame para termos 1 linha por (materia, embedding, metrica)
df_melted = df_exp.melt(
    id_vars=["materia", "embedding"],
    value_vars=metricas_validas,
    var_name="metrica",
    value_name="score"
).dropna(subset=["score"])

# Calcula a posição (Rank 1 = Maior Score) dentro de cada grupo (materia, metrica)
# method='min' garante ranking padrão de competição em caso de empate
df_melted["rank"] = (
    df_melted.groupby(["materia", "metrica"])["score"]
    .rank(ascending=False, method="min")
    .astype(int)
)

# =====================================================================
# 3. TABELA DE FREQUÊNCIA DE POSIÇÕES (TOP 1, TOP 2, TOP 3, ...)
# =====================================================================
# Cria a matriz: Linha = Modelo/Embedding | Coluna = Posição no Rank
df_ranks_count = pd.crosstab(
    index=df_melted["embedding"],
    columns=df_melted["rank"],
    margins=False
)

# Renomeia as colunas para formato amigável (Top 1, Top 2, ...)
df_ranks_count.columns = [f"Top {col}" for col in df_ranks_count.columns]

# Adiciona Rank Médio para ordenação global consistente
rank_medio = df_melted.groupby("embedding")["rank"].mean().rename("Rank Médio")
df_ranks_count["Rank Médio"] = rank_medio.round(2)

# Ordena a tabela pelos mais frequentes no Top 1 e depois pelo melhor Rank Médio
df_ranks_count = df_ranks_count.sort_values(by=[ "Rank Médio", "Top 1"], ascending=[True, False])

# Preenche posições vazias com 0
df_ranks_count = df_ranks_count.fillna(0).astype({col: int for col in df_ranks_count.columns if col != "Rank Médio"})

# =====================================================================
# 4. EXIBIÇÃO DOS RESULTADOS
# =====================================================================
print("=" * 80)
print("DISTRIBUIÇÃO DE POSIÇÕES POR MODELO (Todas as Matérias e Métricas)")
print("=" * 80)
display(df_ranks_count)

DISTRIBUIÇÃO DE POSIÇÕES POR MODELO (Todas as Matérias e Métricas)


,Top 1,Top 2,Top 3,Top 4,Top 5,Top 6,Top 7,Top 8,Top 9,Top 10,Top 11,Top 12,Top 13,Top 14,Top 15,Top 16,Top 17,Top 18,Top 19,Rank Médio
embedding,,,,,,,,,,,,,,,,,,,,
embedding__Octen__Octen_Embedding_8B__texto_preprocessado,4,3,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,2.33
embedding__gemini__gemini_embedding_2__sts__texto_preprocessado,4,1,1,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2.56
embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado,0,4,4,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,2.89
embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado,0,0,2,2,4,0,0,1,0,0,0,0,0,0,0,0,0,0,0,4.67
embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado,0,1,0,4,0,2,1,0,0,0,0,0,1,0,0,0,0,0,0,5.56
embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado,0,0,1,0,1,2,2,1,2,0,0,0,0,0,0,0,0,0,0,6.67
embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto_preprocessado,0,1,0,0,0,2,2,0,1,1,2,0,0,0,0,0,0,0,0,7.67
embedding__openai__text_embedding_3_large__texto_preprocessado,0,0,0,0,0,1,3,2,1,2,0,0,0,0,0,0,0,0,0,8.00
embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado,0,0,0,0,1,0,2,2,1,3,0,0,0,0,0,0,0,0,0,8.22


In [97]:

metricas = [
    "precision@1",
    "r_precision",
    "map",

    "ndcg_all",
]

# 1. Filtrar o recorte desejado
df_subset = df_all[
    (df_all["nivel"] == "tema")
    & (df_all["tipo_texto"] == "texto_preprocessado")
    & (df_all["embedding"].str.contains("gemini_embedding_2"))
].copy()

# 2. Separar e indexar pela matéria
df_raw = df_subset[df_subset["embedding"].str.contains("__raw__")].set_index(
    "materia"
)[metricas]
df_sts = df_subset[df_subset["embedding"].str.contains("__sts__")].set_index(
    "materia"
)[metricas]

# 3. Calcular a matriz de ganho percentual
df_ganho_por_materia = ((df_sts - df_raw) / df_raw) * 100

# Adicionar sufixo nas colunas para clareza
df_ganho_por_materia.columns = [f"{col}_ganho_%" for col in metricas]
print(df_ganho_por_materia)

              precision@1_ganho_%  r_precision_ganho_%  map_ganho_%  \
materia                                                               
PEC_6_2019              16.580311            54.681690    53.501987   
MPV_612_2013             8.527132            27.689788    26.864220   
PLP_68_2024             13.052782            42.475633    49.822437   

              ndcg_all_ganho_%  
materia                         
PEC_6_2019                 NaN  
MPV_612_2013               NaN  
PLP_68_2024          12.391915  


## Análise de correlação (tamanho dos modelos abertos vs desempenho)

In [35]:
embedding_cols_abertos = [
    'embedding__codefuse_ai__F2LLM_v2_14B__texto',
       'embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado',
       'embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado_sem_justificativa',
       'embedding__Octen__Octen_Embedding_8B__texto',
       'embedding__Octen__Octen_Embedding_8B__texto_preprocessado',
       'embedding__Octen__Octen_Embedding_8B__texto_preprocessado_sem_justificativa',
       'embedding__Qwen__Qwen3_Embedding_8B__texto',
       'embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado',
       'embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado_sem_justificativa',
       'embedding__nvidia__llama_embed_nemotron_8b__texto',
       'embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado',
       'embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado_sem_justificativa',
       'embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto',
       'embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto_preprocessado',
       'embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto_preprocessado_sem_justificativa',
       'embedding__joaorobson__harrier_oss_v1_27b__texto',
       'embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado',
       'embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado_sem_justificativa',
       'embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto',
       'embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto_preprocessado',
       'embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__trunc128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__trunc128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__trunc128__texto_preprocessado_sem_justificativa',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto_preprocessado',
       'embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto_preprocessado_sem_justificativa',
       'embedding__josedossantos__bertimbau_tuned__trunc512__texto',
       'embedding__josedossantos__bertimbau_tuned__trunc512__texto_preprocessado',
       'embedding__josedossantos__bertimbau_tuned__trunc512__texto_preprocessado_sem_justificativa',
       'embedding__josedossantos__bertimbau_tuned__meanpool512__texto',
       'embedding__josedossantos__bertimbau_tuned__meanpool512__texto_preprocessado',
       'embedding__josedossantos__bertimbau_tuned__meanpool512__texto_preprocessado_sem_justificativa',
       'embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto',
       'embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado',
       'embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado_sem_justificativa']

In [71]:
import numpy as np
import pandas as pd
from scipy.stats import linregress

# =====================================================
# 1. MAPEAMENTO DE TAMANHOS E DEFINIÇÃO DAS FAIXAS (TIERS)
# =====================================================

MODEL_SIZE = {
    "F2LLM_v2_14B": 14.0,
    "Octen_Embedding_8B": 8.0,
    "Qwen3_Embedding_8B": 8.0,
    "llama_embed_nemotron_8b": 8.0,
    "harrier_oss_v1_27b": 27.0,
    "KaLM_Embedding_Gemma3_12B_2511": 12.0,
    "jina_embeddings_v5_text_small": 0.6,
    "serafim_335m_portuguese_pt_sentence_encoder": 0.335,
    "serafim_335m_portuguese_pt_sentence_encoder_ir": 0.335,
    "bertimbau_tuned": 0.335,
    "ICT_TIME_and_Querit_embedding_v1": 4.0
}

def get_model_size(name):
    for model, size in MODEL_SIZE.items():
        if model in name:
            return size
    return np.nan

def assign_tier(size):
    """Categoriza o modelo por faixa de capacidade em parâmetros."""
    if size < 1.0:
        return "1. Small / Specialized (< 1B)"
    elif size <= 8.0:
        return "2. Mid-scale (4B - 8B)"
    elif size > 8.0:
        return "3. Large Open (12B - 27B)"
    return np.nan

# =====================================================
# 2. PREPARAÇÃO DOS DADOS COM FILTRO (nivel == 'tema')
# =====================================================

df_analise = df_abertos.copy()

# Filtra estritamente o nível original/específico
if "nivel" in df_analise.columns:
    df_analise = df_analise[df_analise["nivel"] == "tema"].copy()

# Mapeia e filtra apenas modelos abertos com tamanho conhecido
df_analise["size_b"] = df_analise["embedding"].apply(get_model_size)
df_analise = df_analise.dropna(subset=["size_b"]).copy()

df_analise["tier"] = df_analise["size_b"].apply(assign_tier)
df_analise["log_size"] = np.log10(df_analise["size_b"])

metrics = ["precision@1", "r_precision", "map"]

# =====================================================
# 3. TABELA 1: MÉDIAS E DESVIOS POR FAIXA (TIERED SUMMARY)
# =====================================================

tier_summary = (
    df_analise
    .groupby(["materia", "tier"])[metrics]
    .agg(["mean", "std"])
    .round(4)
)

print("=" * 70)
print("DESEMPENHO MÉDIO POR FAIXA DE PARÂMETROS (TIERS) - NÍVEL TEMA")
print("=" * 70)
display(tier_summary)

# =====================================================
# 4. TABELA 2: REGRESSÃO LOG-LINEAR (SCALING LAWS)
# =====================================================

reg_results = []

for materia in df_analise["materia"].unique():
    sub = df_analise[df_analise["materia"] == materia]
    
    for metric in metrics:
        slope, intercept, r_value, p_value, std_err = linregress(sub["log_size"], sub[metric])
        
        reg_results.append({
            "materia": materia,
            "metric": metric,
            "slope_beta (ganho por 10x)": round(slope, 4),
            "r_squared (R²)": round(r_value**2, 4),
            "p_value": round(p_value, 4)
        })

df_reg = pd.DataFrame(reg_results)

print("\n" + "=" * 70)
print("REGRESSÃO LOG-LINEAR: MÉTRICA ~ log10(SIZE) - NÍVEL TEMA")
print("=" * 70)
display(df_reg)

DESEMPENHO MÉDIO POR FAIXA DE PARÂMETROS (TIERS) - NÍVEL TEMA


precision@1         r_precision  \
                                                  mean     std        mean   
materia      tier                                                            
MPV_612_2013 1. Small / Specialized (< 1B)      0.7023  0.0356      0.4549   
             2. Mid-scale (4B - 8B)             0.7709  0.0065      0.5819   
             3. Large Open (12B - 27B)          0.7728  0.0465      0.5970   
PEC_6_2019   1. Small / Specialized (< 1B)      0.7202  0.0649      0.4245   
             2. Mid-scale (4B - 8B)             0.8267  0.0390      0.6128   
             3. Large Open (12B - 27B)          0.8258  0.0303      0.6003   
PLP_68_2024  1. Small / Specialized (< 1B)      0.7297  0.0486      0.3951   
             2. Mid-scale (4B - 8B)             0.8151  0.0043      0.5471   
             3. Large Open (12B - 27B)          0.8079  0.0182      0.5320   

                                                       map          
                                               std    mean     std  
materia      tier                                                   
MPV_612_2013 1. Small / Specialized (< 1B)  0.0330  0.4814  0.0332  
             2. Mid-scale (4B - 8B)         0.0423  0.6268  0.0426  
             3. Large Open (12B - 27B)      0.0187  0.6452  0.0196  
PEC_6_2019   1. Small / Specialized (< 1B)  0.0934  0.4649  0.0969  
             2. Mid-scale (4B - 8B)         0.0458  0.6605  0.0508  
             3. Large Open (12B - 27B)      0.0804  0.6432  0.0840  
PLP_68_2024  1. Small / Specialized (< 1B)  0.0752  0.4075  0.0855  
             2. Mid-scale (4B - 8B)         0.0194  0.5781  0.0231  
             3. Large Open (12B - 27B)      0.0612  0.5599  0.0716


REGRESSÃO LOG-LINEAR: MÉTRICA ~ log10(SIZE) - NÍVEL TEMA


,materia,metric,slope_beta (ganho por 10x),r_squared (R²),p_value
0,PEC_6_2019,precision@1,0.0764,0.6295,0.0007
1,PEC_6_2019,r_precision,0.1289,0.6727,0.0003
2,PEC_6_2019,map,0.1324,0.6622,0.0004
3,MPV_612_2013,precision@1,0.0476,0.5997,0.0011
4,MPV_612_2013,r_precision,0.0920,0.8596,0.0000
5,MPV_612_2013,map,0.1052,0.8857,0.0000
6,PLP_68_2024,precision@1,0.0562,0.6238,0.0008
7,PLP_68_2024,r_precision,0.1010,0.6672,0.0004
8,PLP_68_2024,map,0.1131,0.6573,0.0004


In [38]:
df_abertos = df_all[(df_all.tipo_texto == "texto_preprocessado") & (df_all.nivel == "tema") & (df_all.embedding.isin(embedding_cols_abertos))].copy()


In [37]:
df_all["size"] = df_all.embedding.apply(get_model_size)

In [40]:
from scipy.stats import spearmanr
import pandas as pd

metrics = ["precision@1", "r_precision", "map"]

results = []

for materia in df_abertos["materia"].unique():
    sub = df_abertos[df_abertos["materia"] == materia]
    
    for metric in metrics:
        rho, p = spearmanr(sub["size"], sub[metric])
        
        results.append({
            "materia": materia,
            "metric": metric,
            "spearman_rho": rho,
            "p_value": p
        })

df_corr = pd.DataFrame(results)
df_corr

,materia,metric,spearman_rho,p_value
0,PEC_6_2019,precision@1,0.769647,0.001286
1,PEC_6_2019,r_precision,0.822869,0.000301
2,PEC_6_2019,map,0.822869,0.000301
3,MPV_612_2013,precision@1,0.764483,0.001451
4,MPV_612_2013,r_precision,0.878033,0.000036
5,MPV_612_2013,map,0.887227,0.000023
6,PLP_68_2024,precision@1,0.790130,0.000772
7,PLP_68_2024,r_precision,0.832063,0.000223
8,PLP_68_2024,map,0.845854,0.000138


In [147]:
from scipy.stats import spearmanr

metrics = ["precision@1", "r_precision", "map"]

for m in metrics:
    rho, p = spearmanr(df_abertos["size"], df_abertos[m])
    print(f"\nMetric: {m}")
    print(f"Spearman rho = {rho:.3f}")
    print(f"p-value = {p:.4f}")


Metric: precision@1
Spearman rho = 0.758
p-value = 0.0000

Metric: r_precision
Spearman rho = 0.821
p-value = 0.0000

Metric: map
Spearman rho = 0.805
p-value = 0.0000


## Avaliação da segmentação do texto

In [81]:
df_all[(df_all["nivel"] == "tema") & (df_all.embedding.str.contains("bm25"))]

,nivel,embedding,precision@1,mrr,r_precision,map,materia,ndcg@5,ndcg@10,ndcg_all,tipo_texto,familia
15,tema,retrieval__bm25l__texto_preprocessado__preprocess,0.734848,0.806532,0.511232,0.544970,PEC_6_2019,NaN,NaN,NaN,texto_preprocessado,bm25
17,tema,retrieval__bm25l__texto_preprocessado__raw,0.738636,0.804027,0.490757,0.523457,PEC_6_2019,NaN,NaN,NaN,texto_preprocessado,bm25
23,tema,retrieval__bm25l__texto__preprocess,0.643939,0.734191,0.435075,0.481518,PEC_6_2019,NaN,NaN,NaN,texto,bm25
27,tema,retrieval__bm25l__texto__raw,0.685606,0.767435,0.442710,0.477607,PEC_6_2019,NaN,NaN,NaN,texto,bm25
30,tema,retrieval__bm25l__texto_preprocessado_sem_just...,0.753788,0.805227,0.436517,0.471245,PEC_6_2019,NaN,NaN,NaN,texto_preprocessado_sem_justificativa,bm25
41,tema,retrieval__bm25l__texto_preprocessado_sem_just...,0.704545,0.768507,0.405019,0.432198,PEC_6_2019,NaN,NaN,NaN,texto_preprocessado_sem_justificativa,bm25
69,tema,retrieval__bm25l__texto_preprocessado__preprocess,0.782123,0.844635,0.560455,0.596075,MPV_612_2013,NaN,NaN,NaN,texto_preprocessado,bm25
73,tema,retrieval__bm25l__texto__preprocess,0.759777,0.827957,0.542672,0.571026,MPV_612_2013,NaN,NaN,NaN,texto,bm25
77,tema,retrieval__bm25l__texto_preprocessado__raw,0.759777,0.830662,0.516099,0.543362,MPV_612_2013,NaN,NaN,NaN,texto_preprocessado,bm25
79,tema,retrieval__bm25l__texto__raw,0.737430,0.813257,0.510542,0.533465,MPV_612_2013,NaN,NaN,NaN,texto,bm25


In [79]:
(
    df_all[df_all["nivel"] == "tema"]
        .groupby(["materia", "tipo_texto","familia"])[metricas]
    .agg(["mean", "std"])
)

precision@1  \
                                                                       mean   
materia      tipo_texto                            familia                    
MPV_612_2013 texto                                 bm25            0.748603   
                                                   multilingual    0.770950   
                                                   proprietary     0.705773   
                                                   specialized     0.551210   
             texto_preprocessado                   bm25            0.770950   
                                                   multilingual    0.770251   
                                                   proprietary     0.757914   
                                                   specialized     0.692737   
             texto_preprocessado_sem_justificativa bm25            0.737430   
                                                   multilingual    0.724162   
                                                   proprietary     0.793296   
                                                   specialized     0.620112   
PEC_6_2019   texto                                 bm25            0.664773   
                                                   multilingual    0.768466   
                                                   proprietary     0.688131   
                                                   specialized     0.611111   
             texto_preprocessado                   bm25            0.736742   
                                                   multilingual    0.830019   
                                                   proprietary     0.794192   
                                                   specialized     0.697601   
             texto_preprocessado_sem_justificativa bm25            0.729167   
                                                   multilingual    0.724432   
                                                   proprietary     0.758838   
                                                   specialized     0.614268   
PLP_68_2024  texto                                 bm25            0.778922   
                                                   multilingual    0.786648   
                                                   proprietary     0.719493   
                                                   specialized     0.636644   
             texto_preprocessado                   bm25            0.792657   
                                                   multilingual    0.811609   
                                                   proprietary     0.793626   
                                                   specialized     0.716499   
             texto_preprocessado_sem_justificativa bm25            0.805335   
                                                   multilingual    0.784073   
                                                   proprietary     0.811763   
                                                   specialized     0.685420   

                                                                           \
                                                                      std   
materia      tipo_texto                            familia                  
MPV_612_2013 texto                                 bm25          0.015801   
                                                   multilingual  0.020689   
                                                   proprietary   0.099988   
                                                   specialized   0.152573   
             texto_preprocessado                   bm25          0.015801   
                                                   multilingual  0.025590   
                                                   proprietary   0.032734   
                                                   specialized   0.027369   
             texto_preprocessado_sem_justificativa bm25          0.007901   
                                                   multili

In [65]:
import pandas as pd
import numpy as np

# 1. Filtra o nível 'tema' e aplica o mapeamento de família
df_analise = df_all[df_all["nivel"] == "tema"].copy()
df_analise["familia"] = df_analise["embedding"].apply(get_model_family)

# 2. Calcula as médias por matéria, família e tipo de texto
metricas = ["precision@1", "r_precision", "map"]
df_medias = (
    df_analise
    .groupby(["materia", "familia", "tipo_texto"])[metricas]
    .mean()
    .reset_index()
)

# 3. Separa o Baseline (Texto Bruto) e os Tratamentos
df_bruto = df_medias[df_medias["tipo_texto"] == "texto"].copy()
df_alt_just = df_medias[df_medias["tipo_texto"] == "texto_preprocessado"].copy()
df_sem_just = df_medias[df_medias["tipo_texto"] == "texto_preprocessado_sem_justificativa"].copy()

# 4. Faz o merge pareado por Matéria e Família
m_just = pd.merge(df_alt_just, df_bruto, on=["materia", "familia"], suffixes=("_alt_just", "_bruto"))
m_sem = pd.merge(df_sem_just, df_bruto, on=["materia", "familia"], suffixes=("_sem_just", "_bruto"))

# 5. Constrói o relatório detalhado de ganhos (Deltas)
registros = []

for materia in df_analise["materia"].unique():
    for fam in df_analise["familia"].dropna().unique():
        sub_j = m_just[(m_just["materia"] == materia) & (m_just["familia"] == fam)]
        sub_s = m_sem[(m_sem["materia"] == materia) & (m_sem["familia"] == fam)]
        
        if sub_j.empty or sub_s.empty:
            continue
            
        for metrica in metricas:
            b = sub_j[f"{metrica}_bruto"].values[0]
            j = sub_j[f"{metrica}_alt_just"].values[0]
            s = sub_s[f"{metrica}_sem_just"].values[0]
            
            delta_just = j - b
            pct_just = (delta_just / b) * 100 if b != 0 else 0
            
            delta_sem = s - b
            pct_sem = (delta_sem / b) * 100 if b != 0 else 0
            
            registros.append({
                "Matéria": materia,
                "Família": fam,
                "Métrica": metrica,
                "Texto Bruto": round(b, 4),
                "Alt + Just": round(j, 4),
                "Δ (Alt+Just)": round(delta_just, 4),
                "% Ganho (Alt+Just)": f"{pct_just:+.2f}%",
                "Alt Pura": round(s, 4),
                "Δ (Alt Pura)": round(delta_sem, 4),
                "% Ganho (Alt Pura)": f"{pct_sem:+.2f}%"
            })

df_ganhos = pd.DataFrame(registros)

# Ordena para facilitar a inspeção
df_ganhos = df_ganhos.sort_values(by=["Matéria", "Família", "Métrica"])

# Exibe o resultado completo
display(df_ganhos)

,Matéria,Família,Métrica,Texto Bruto,Alt + Just,Δ (Alt+Just),% Ganho (Alt+Just),Alt Pura,Δ (Alt Pura),% Ganho (Alt Pura)
20,MPV_612_2013,bm25,map,0.5522,0.5697,0.0175,+3.16%,0.5067,-0.0455,-8.25%
18,MPV_612_2013,bm25,precision@1,0.7486,0.7709,0.0223,+2.99%,0.7374,-0.0112,-1.49%
19,MPV_612_2013,bm25,r_precision,0.5266,0.5383,0.0117,+2.22%,0.4679,-0.0587,-11.15%
14,MPV_612_2013,multilingual,map,0.5761,0.6242,0.0481,+8.35%,0.5104,-0.0657,-11.40%
12,MPV_612_2013,multilingual,precision@1,0.7709,0.7703,-0.0007,-0.09%,0.7242,-0.0468,-6.07%
13,MPV_612_2013,multilingual,r_precision,0.5335,0.5802,0.0468,+8.77%,0.4679,-0.0655,-12.28%
17,MPV_612_2013,proprietary,map,0.5192,0.6044,0.0851,+16.39%,0.5619,0.0427,+8.21%
15,MPV_612_2013,proprietary,precision@1,0.7058,0.7579,0.0521,+7.39%,0.7933,0.0875,+12.40%
16,MPV_612_2013,proprietary,r_precision,0.4852,0.5670,0.0817,+16.84%,0.5276,0.0423,+8.73%
23,MPV_612_2013,specialized,map,0.3971,0.4698,0.0727,+18.32%,0.4144,0.0173,+4.35%


In [66]:
# 1. Garante a conversão da coluna de % para float numérico
df_ganhos["pct_num_Alt_Just"] = (
    df_ganhos["% Ganho (Alt+Just)"]
    .str.rstrip("%")
    .astype(float)
)
df_ganhos["pct_num_Alt_SemJust"] = (
    df_ganhos["% Ganho (Alt Pura)"]
    .str.rstrip("%")
    .astype(float)
)

# 2. Agrupa por Família e Métrica agregando médias
df_ganhos_familia = (
    df_ganhos
    .groupby(["Família", "Métrica"])[
        ["Texto Bruto", "Alt + Just", "Δ (Alt+Just)", "pct_num_Alt_Just", 
         "Alt Pura", "Δ (Alt Pura)", "pct_num_Alt_SemJust"]
    ]
    .mean()
    .round(4)
    .reset_index()
)

# 3. Formata as colunas percentuais de volta para string com sinal
df_ganhos_familia["% Médio (Alt+Just)"] = df_ganhos_familia["pct_num_Alt_Just"].apply(lambda x: f"{x:+.2f}%")
df_ganhos_familia["% Médio (Alt Pura)"] = df_ganhos_familia["pct_num_Alt_SemJust"].apply(lambda x: f"{x:+.2f}%")

# 4. Seleciona e organiza as colunas finais
cols_exibicao = [
    "Família", "Métrica", 
    "Texto Bruto", "Alt + Just", "Δ (Alt+Just)", "% Médio (Alt+Just)",
    "Alt Pura", "Δ (Alt Pura)", "% Médio (Alt Pura)"
]
df_ganhos_familia = df_ganhos_familia[cols_exibicao].sort_values(by=["Família", "Métrica"])

display(df_ganhos_familia)

,Família,Métrica,Texto Bruto,Alt + Just,Δ (Alt+Just),% Médio (Alt+Just),Alt Pura,Δ (Alt Pura),% Médio (Alt Pura)
0,bm25,map,0.5149,0.5496,0.0348,+6.95%,0.4953,-0.0195,-3.73%
1,bm25,precision@1,0.7308,0.7668,0.0360,+5.19%,0.7573,0.0265,+3.86%
2,bm25,r_precision,0.4831,0.5186,0.0356,+7.72%,0.4658,-0.0173,-3.38%
3,multilingual,map,0.5660,0.6149,0.0489,+8.56%,0.4754,-0.0906,-15.86%
4,multilingual,precision@1,0.7753,0.8040,0.0286,+3.70%,0.7442,-0.0311,-4.04%
5,multilingual,r_precision,0.5291,0.5749,0.0458,+8.60%,0.4455,-0.0837,-15.67%
6,proprietary,map,0.4953,0.5703,0.0750,+15.07%,0.5185,0.0232,+4.72%
7,proprietary,precision@1,0.7045,0.7819,0.0774,+11.03%,0.7880,0.0835,+11.83%
8,proprietary,r_precision,0.4636,0.5358,0.0722,+15.55%,0.4890,0.0254,+5.41%
9,specialized,map,0.3659,0.4293,0.0633,+17.20%,0.3451,-0.0208,-6.01%


In [67]:
# 1. Garante a conversão das colunas percentuais para float
df_ganhos["pct_num_Alt_Just"] = (
    df_ganhos["% Ganho (Alt+Just)"]
    .str.rstrip("%")
    .astype(float)
)
df_ganhos["pct_num_Alt_SemJust"] = (
    df_ganhos["% Ganho (Alt Pura)"]
    .str.rstrip("%")
    .astype(float)
)

# 2. Agrupa apenas por Métrica agregando médias e desvios
colunas_num = [
    "Texto Bruto", "Alt + Just", "Δ (Alt+Just)", "pct_num_Alt_Just",
    "Alt Pura", "Δ (Alt Pura)", "pct_num_Alt_SemJust"
]

df_resumo_metrica = (
    df_ganhos
    .groupby("Métrica")[colunas_num]
    .agg(["mean", "std"])
    .round(4)
)

# 3. Cria o DataFrame consolidado ordenado pelo maior ganho
df_ganhos_por_metrica = pd.DataFrame(index=df_resumo_metrica.index)

df_ganhos_por_metrica["Texto Bruto"] = df_resumo_metrica[("Texto Bruto", "mean")]
df_ganhos_por_metrica["Alt + Just"] = df_resumo_metrica[("Alt + Just", "mean")]
df_ganhos_por_metrica["Δ Médio (Alt+Just)"] = df_resumo_metrica[("Δ (Alt+Just)", "mean")]
df_ganhos_por_metrica["% Ganho Médio (Alt+Just)"] = df_resumo_metrica[("pct_num_Alt_Just", "mean")]
df_ganhos_por_metrica["Alt Pura"] = df_resumo_metrica[("Alt Pura", "mean")]
df_ganhos_por_metrica["Δ Médio (Alt Pura)"] = df_resumo_metrica[("Δ (Alt Pura)", "mean")]
df_ganhos_por_metrica["% Ganho Médio (Alt Pura)"] = df_resumo_metrica[("pct_num_Alt_SemJust", "mean")]

# Ordena pelo maior ganho percentual obtido com Alt + Just
df_ganhos_por_metrica = df_ganhos_por_metrica.sort_values(
    by="% Ganho Médio (Alt+Just)", 
    ascending=False
).reset_index()

# 4. Formata as colunas percentuais para exibição
df_ganhos_por_metrica["% Ganho Médio (Alt+Just)"] = df_ganhos_por_metrica["% Ganho Médio (Alt+Just)"].apply(lambda x: f"{x:+.2f}%")
df_ganhos_por_metrica["% Ganho Médio (Alt Pura)"] = df_ganhos_por_metrica["% Ganho Médio (Alt Pura)"].apply(lambda x: f"{x:+.2f}%")

display(df_ganhos_por_metrica)

,Métrica,Texto Bruto,Alt + Just,Δ Médio (Alt+Just),% Ganho Médio (Alt+Just),Alt Pura,Δ Médio (Alt Pura),% Ganho Médio (Alt Pura)
0,r_precision,0.4553,0.5084,0.0531,+12.21%,0.4311,-0.0242,-5.04%
1,map,0.4855,0.5410,0.0555,+11.94%,0.4586,-0.0269,-5.22%
2,precision@1,0.7026,0.7637,0.0612,+9.35%,0.7324,0.0298,+4.64%


In [68]:
df_mat = df_all[(df_all["nivel"] == "tema") & (df_all["materia"] == "PLP_68_2024")]

nomes_bruto = df_mat[df_mat["tipo_texto"] == "texto"].sort_values("embedding")["embedding"].values
nomes_alt = df_mat[df_mat["tipo_texto"] == "texto_preprocessado"].sort_values("embedding")["embedding"].values

for i, (b, a) in enumerate(zip(nomes_bruto, nomes_alt)):
    print(f"{i:02d} | Bruto: {b}\n   | Alt  : {a}\n")

00 | Bruto: embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto
   | Alt  : embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado

01 | Bruto: embedding__Octen__Octen_Embedding_8B__texto
   | Alt  : embedding__Octen__Octen_Embedding_8B__texto_preprocessado

02 | Bruto: embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto
   | Alt  : embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__meanpool128__texto_preprocessado

03 | Bruto: embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto
   | Alt  : embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder__trunc128__texto_preprocessado

04 | Bruto: embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto
   | Alt  : embedding__PORTULAN__serafim_335m_portuguese_pt_sentence_encoder_ir__meanpool128__texto_preprocessado

05 | Bruto: embedding__PORTULAN__serafim_335m_portuguese_pt_sen

In [75]:
df_all

,nivel,embedding,precision@1,mrr,r_precision,map,materia,ndcg@5,ndcg@10,ndcg_all,tipo_texto,familia
0,tema,embedding__Octen__Octen_Embedding_8B__texto,0.867424,0.895425,0.669929,0.716369,PEC_6_2019,NaN,NaN,NaN,texto,multilingual
1,tema,embedding__Octen__Octen_Embedding_8B__texto_pr...,0.867424,0.897080,0.658753,0.713185,PEC_6_2019,NaN,NaN,NaN,texto_preprocessado,multilingual
2,tema,embedding__joaorobson__harrier_oss_v1_27b__tex...,0.856061,0.894553,0.653239,0.699050,PEC_6_2019,NaN,NaN,NaN,texto_preprocessado,multilingual
3,tema,embedding__Qwen__Qwen3_Embedding_8B__texto_pre...,0.852273,0.893137,0.645340,0.694852,PEC_6_2019,NaN,NaN,NaN,texto_preprocessado,multilingual
4,tema,embedding__Qwen__Qwen3_Embedding_8B__texto,0.856061,0.893263,0.640187,0.692511,PEC_6_2019,NaN,NaN,NaN,texto,multilingual
...,...,...,...,...,...,...,...,...,...,...,...,...
280,tema_nivel_2,embedding__josedossantos__bertimbau_tuned__mea...,0.741406,0.788396,0.239505,0.238021,PLP_68_2024,NaN,NaN,NaN,texto_preprocessado_sem_justificativa,specialized
281,tema,embedding__PORTULAN__serafim_335m_portuguese_p...,0.511886,0.608913,0.240404,0.232702,PLP_68_2024,0.496227,0.441459,0.668565,texto,specialized
282,tema_nivel_2,embedding__PORTULAN__serafim_335m_portuguese_p...,0.662391,0.738744,0.223799,0.211263,PLP_68_2024,NaN,NaN,NaN,texto,specialized
283,tema,embedding__PORTULAN__serafim_335m_portuguese_p...,0.486001,0.570829,0.206915,0.203111,PLP_68_2024,0.460896,0.404061,0.650428,texto,specialized


In [73]:
import gc
import scipy.stats as stats
import pandas as pd
import numpy as np

# =====================================================================
# 1. CONSOLIDAÇÃO DAS ESTATÍSTICAS DESCRITIVAS (MÉDIA E DESVIO PADRÃO)
# =====================================================================
print("--- Calculando Média e Desvio Padrão por Tratamento ---")

print("\n" + "="*70)
print("INICIANDO ANÁLISE ESTATÍSTICA COMPLETA")
print("="*70)

# =====================================================================
# 2. CONSOLIDAÇÃO DAS ESTATÍSTICAS DESCRITIVAS (MÉDIA E DESVIO PADRÃO)
# =====================================================================
print("\n--- 1. Calculando Média e Desvio Padrão por Tratamento ---")

df_estatisticas = (
    df_all[df_all["nivel"] == "tema"]
    .groupby(["materia", "tipo_texto"])[metricas]
    .agg(["mean", "std"])
)

# Simplifica o multi-index das colunas para 'metrica_mean' e 'metrica_std'
df_estatisticas.columns = [f"{metric}_{stat}" for metric, stat in df_estatisticas.columns]
display(df_estatisticas)


# =====================================================================
# 3. TESTES DE SIGNIFICÂNCIA ESTATÍSTICA (MELHORA EM RELAÇÃO AO BRUTO)
# =====================================================================
print("\n--- 2. Executando Testes de Wilcoxon (Melhora vs. Bruto) ---")

# Filtra apenas o nível 'tema'
df_filtrado = df_all[df_all["nivel"] == "tema"].copy()
lista_materias = df_filtrado["materia"].unique()
registros_testes = []

for materia in lista_materias:
    df_mat = df_filtrado[df_filtrado["materia"] == materia]
    
    # Isola o baseline fixo: Texto Bruto
    df_bruto = df_mat[df_mat["tipo_texto"] == "texto"]
    
    if df_bruto.empty:
        print(f"[Aviso] Matéria {materia} não possui dados para o tipo 'texto' (Bruto).")
        continue
        
    for metric in metricas:
        # Vetor de pontuações do Baseline Bruto (tamanho 10, um para cada modelo)
        scores_bruto = df_bruto.sort_values("embedding")[metric].values
        
        # -----------------------------------------------------------------
        # Teste 1: Pré-processado COMPLETO (Alt. + Just.) SUPEROU o Bruto?
        # -----------------------------------------------------------------
        df_alt_just = df_mat[df_mat["tipo_texto"] == "texto_preprocessado"]
        if not df_alt_just.empty:
            scores_alt_just = df_alt_just.sort_values("embedding")[metric].values
            
            if len(scores_alt_just) == len(scores_bruto):
                # 'greater' testa se: scores_alt_just > scores_bruto
                _, p_val_alt_just = stats.wilcoxon(scores_alt_just, scores_bruto, alternative="greater")
            else:
                p_val_alt_just = np.nan
        else:
            p_val_alt_just = np.nan
            
        # -----------------------------------------------------------------
        # Teste 2: Pré-processado SEM Justificativa (Alt. Pura) SUPEROU o Bruto?
        # -----------------------------------------------------------------
        df_sem_j = df_mat[df_mat["tipo_texto"] == "texto_preprocessado_sem_justificativa"]
        if not df_sem_j.empty:
            scores_sem_j = df_sem_j.sort_values("embedding")[metric].values
            
            if len(scores_sem_j) == len(scores_bruto):
                # 'greater' testa se: scores_sem_j > scores_bruto
                _, p_val_sem_j = stats.wilcoxon(scores_sem_j, scores_bruto, alternative="greater")
            else:
                p_val_sem_j = np.nan
        else:
            p_val_sem_j = np.nan
            
        # Guarda os p-values calculados para esta métrica nesta matéria
        registros_testes.append({
            "materia": materia,
            "metrica": metric,
            "p_val_Alt_Just_vs_Bruto": p_val_alt_just,
            "p_val_Alt_SemJust_vs_Bruto": p_val_sem_j
        })

# Consolida os resultados em DataFrame
df_p_values = pd.DataFrame(registros_testes)

# Define o nível de significância rigoroso do seu paper (Alpha = 0.05)
ALPHA_CORTE = 0.05

# Cria as colunas de veredito se houve ganho ou não em relação ao bruto
df_p_values["Melhorou_Alt_Just?"] = df_p_values["p_val_Alt_Just_vs_Bruto"].apply(
    lambda p: "SIM (*)" if p < ALPHA_CORTE else "Não"
)
df_p_values["Melhorou_Alt_SemJust?"] = df_p_values["p_val_Alt_SemJust_vs_Bruto"].apply(
    lambda p: "SIM (*)" if p < ALPHA_CORTE else "Não"
)

# Reorganiza as colunas para melhor leitura na tela
colunas_ordenadas = [
    "materia", "metrica", 
    "p_val_Alt_Just_vs_Bruto", "Melhorou_Alt_Just?",
    "p_val_Alt_SemJust_vs_Bruto", "Melhorou_Alt_SemJust?"
]
df_p_values = df_p_values[colunas_ordenadas]

print("\n>>> TABELA DE VEREDITO FINAL (Confiança de 95%) <<<")
display(df_p_values)

# Limpeza de memória final
#del df_filtrado, registros_testes
gc.collect()

--- Calculando Média e Desvio Padrão por Tratamento ---

INICIANDO ANÁLISE ESTATÍSTICA COMPLETA

--- 1. Calculando Média e Desvio Padrão por Tratamento ---


precision@1_mean  \
materia      tipo_texto                                                
MPV_612_2013 texto                                          0.688915   
             texto_preprocessado                            0.743899   
             texto_preprocessado_sem_justificativa          0.703617   
PEC_6_2019   texto                                          0.695175   
             texto_preprocessado                            0.772727   
             texto_preprocessado_sem_justificativa          0.695574   
PLP_68_2024  texto                                          0.727862   
             texto_preprocessado                            0.776740   
             texto_preprocessado_sem_justificativa          0.759530   

                                                    precision@1_std  \
materia      tipo_texto                                               
MPV_612_2013 texto                                         0.132335   
             texto_preprocessado                           0.043484   
             texto_preprocessado_sem_justificativa         0.070603   
PEC_6_2019   texto                                         0.114645   
             texto_preprocessado                           0.067665   
             texto_preprocessado_sem_justificativa         0.069366   
PLP_68_2024  texto                                         0.096105   
             texto_preprocessado                           0.050209   
             texto_preprocessado_sem_justificativa         0.056068   

                                                    r_precision_mean  \
materia      tipo_texto                                                
MPV_612_2013 texto                                          0.475640   
             texto_preprocessado                            0.530560   
             texto_preprocessado_sem_justificativa          0.451130   
PEC_6_2019   texto                                          0.451191   
             texto_preprocessado                            0.517141   
             texto_preprocessado_sem_justificativa          0.373280   
PLP_68_2024  texto                                          0.441065   
             texto_preprocessado                            0.479191   
             texto_preprocessado_sem_justificativa          0.424223   

                                                    r_precision_std  map_mean  \
materia      tipo_texto                                                         
MPV_612_2013 texto                                         0.107215  0.508090   
             texto_preprocessado                           0.071151  0.566601   
             texto_preprocessado_sem_justificativa         0.054461  0.487839   
PEC_6_2019   texto                                         0.133151  0.490154   
             texto_preprocessado                           0.109905  0.559386   
             texto_preprocessado_sem_justificativa         0.074585  0.403788   
PLP_68_2024  texto                                         0.104781  0.460573   
             texto_preprocessado                           0.088173  0.501104   
             texto_preprocessado_sem_justificativa         0.085450  0.437816   

                                                     map_std  
materia      tipo_texto                                       
MPV_612_2013 texto                                  0.117048  
             texto_preprocessado                    0.079127  
             texto_preprocessado_sem_justificativa  0.058989  
PEC_6_2019   texto                                  0.138252  
             texto_preprocessado                    0.114488  
             texto_preprocessado_sem_justificativa  0.076807  
PLP_68_2024  texto                                  0.117774  
             texto_preprocessado                    0.099901  
             texto_preprocessado_sem_justificativa  0.093119


--- 2. Executando Testes de Wilcoxon (Melhora vs. Bruto) ---

>>> TABELA DE VEREDITO FINAL (Confiança de 95%) <<<


,materia,metrica,p_val_Alt_Just_vs_Bruto,Melhorou_Alt_Just?,p_val_Alt_SemJust_vs_Bruto,Melhorou_Alt_SemJust?
0,PEC_6_2019,precision@1,0.000176,SIM (*),0.540096,Não
1,PEC_6_2019,r_precision,0.000013,SIM (*),0.996428,Não
2,PEC_6_2019,map,0.000006,SIM (*),0.997335,Não
3,MPV_612_2013,precision@1,0.009158,SIM (*),0.842879,Não
4,MPV_612_2013,r_precision,0.000036,SIM (*),0.938406,Não
5,MPV_612_2013,map,0.000010,SIM (*),0.927656,Não
6,PLP_68_2024,precision@1,0.000107,SIM (*),0.092068,Não
7,PLP_68_2024,r_precision,0.000010,SIM (*),0.853251,Não
8,PLP_68_2024,map,0.000010,SIM (*),0.902184,Não


364

## Avaliação dos níveis temáticos

In [77]:
# =====================================================================
# CONFIGURAÇÃO
# =====================================================================
metricas = ["precision@1", "r_precision", "map"]

# Mapeamento dos níveis internos do DataFrame para os nomes do artigo
map_niveis = {
    "tema": "Original Theme",
    "tema_nivel_2": "Intermediate Theme",
    "tema_macro": "General Theme"
}
ordem_niveis = ["Original Theme", "Intermediate Theme", "General Theme"]

# Filtra apenas a matéria PLP_68_2024 na versão padrão (Changes + Justification)
df_plp = df_all[
    (df_all["materia"] == "PLP_68_2024") & 
    (df_all["tipo_texto"] == "texto_preprocessado")
].copy()

df_plp["Level"] = df_plp["nivel"].map(map_niveis)

# =====================================================================
# 1. ESTATÍSTICAS DESCRITIVAS (MEAN +- STD)
# =====================================================================
print("=" * 70)
print("ESTATÍSTICAS DESCRITIVAS POR NÍVEL TEMÁTICO (PLP 68/2024)")
print("=" * 70)

df_estatisticas_plp = (
    df_plp.groupby("Level")[metricas]
    .agg(["mean", "std"])
    .reindex(ordem_niveis)
)

display(df_estatisticas_plp)

# =====================================================================
# 2. TESTES DE WILCOXON (INTERMEDIATE E GENERAL VS. ORIGINAL THEME)
# =====================================================================
print("\n" + "=" * 70)
print("TESTES DE WILCOXON PAREADOS (vs. Original Theme)")
print("=" * 70)

# Isola o baseline (Original Theme)
df_orig = df_plp[df_plp["Level"] == "Original Theme"]
niveis_comparacao = ["Intermediate Theme", "General Theme"]
registros_testes_niveis = []

for nivel in niveis_comparacao:
    df_nivel = df_plp[df_plp["Level"] == nivel]
    
    for metric in metricas:
        # Garante o alinhamento exato dos modelos pareados
        scores_orig = df_orig.sort_values("embedding")[metric].values
        scores_nivel = df_nivel.sort_values("embedding")[metric].values
        
        if len(scores_orig) == len(scores_nivel) and len(scores_orig) > 0:
            # Teste unilateral: verifica ganho em relação ao tema original
            stat, p_val_greater = stats.wilcoxon(scores_nivel, scores_orig, alternative="greater")
            # Teste bilateral/unilateral geral para checar significância
            _, p_val_less = stats.wilcoxon(scores_nivel, scores_orig, alternative="less")
        else:
            p_val_greater = np.nan
            p_val_less = np.nan
            
        registros_testes_niveis.append({
            "Level": nivel,
            "Metric": metric,
            "Mean_Orig": np.mean(scores_orig),
            "Mean_Level": np.mean(scores_nivel),
            "p_val_greater": p_val_greater,
            "p_val_less": p_val_less,
            "p_val_min": min(p_val_greater, p_val_less) if pd.notnull(p_val_greater) else np.nan
        })

df_wilcoxon_niveis = pd.DataFrame(registros_testes_niveis)
df_wilcoxon_niveis["Significante (p < 0.05)?"] = df_wilcoxon_niveis["p_val_greater"].apply(
    lambda p: "SIM (*)" if p < 0.05 else "Não"
)

display(df_wilcoxon_niveis)

# =====================================================================
# 3. EXIBIÇÃO NO FORMATO IDÊNTICO À TABELA LATEX
# =====================================================================
print("\n" + "=" * 70)
print("LINHAS FORMATADAS PARA A TABELA LATEX")
print("=" * 70)

for nivel in ordem_niveis:
    row_str = f"{nivel} & "
    vals = []
    for m in metricas:
        mean_val = df_estatisticas_plp.loc[nivel, (m, "mean")]
        std_val = df_estatisticas_plp.loc[nivel, (m, "std")]
        
        # Identifica se houve significância estatística de aumento (apenas P@1)
        sig = ""
        if nivel != "Original Theme":
            p = df_wilcoxon_niveis[
                (df_wilcoxon_niveis["Level"] == nivel) & 
                (df_wilcoxon_niveis["Metric"] == m)
            ]["p_val_greater"].values[0]
            if p < 0.05:
                sig = "^{*}"
        
        vals.append(f"${mean_val:.3f} \\pm {std_val:.3f}{sig}$")
        
    print(row_str + " &\n".join(vals) + r" \\" + "\n")

gc.collect()

ESTATÍSTICAS DESCRITIVAS POR NÍVEL TEMÁTICO (PLP 68/2024)


precision@1           r_precision                 map  \
                          mean       std        mean       std      mean   
Level                                                                      
Original Theme        0.776740  0.050209    0.479191  0.088173  0.501104   
Intermediate Theme    0.868489  0.039566    0.418969  0.081972  0.435049   
General Theme         0.903437  0.028257    0.359648  0.051111  0.375863   

                              
                         std  
Level                         
Original Theme      0.099901  
Intermediate Theme  0.094685  
General Theme       0.058235


TESTES DE WILCOXON PAREADOS (vs. Original Theme)


,Level,Metric,Mean_Orig,Mean_Level,p_val_greater,p_val_less,p_val_min,Significante (p < 0.05)?
0,Intermediate Theme,precision@1,0.776740,0.868489,0.000002,1.000000,0.000002,SIM (*)
1,Intermediate Theme,r_precision,0.479191,0.418969,1.000000,0.000002,0.000002,Não
2,Intermediate Theme,map,0.501104,0.435049,1.000000,0.000002,0.000002,Não
3,General Theme,precision@1,0.776740,0.903437,0.000066,0.999934,0.000066,SIM (*)
4,General Theme,r_precision,0.479191,0.359648,1.000000,0.000002,0.000002,Não
5,General Theme,map,0.501104,0.375863,1.000000,0.000002,0.000002,Não



LINHAS FORMATADAS PARA A TABELA LATEX
Original Theme & $0.777 \pm 0.050$ &
$0.479 \pm 0.088$ &
$0.501 \pm 0.100$ \\

Intermediate Theme & $0.868 \pm 0.040^{*}$ &
$0.419 \pm 0.082$ &
$0.435 \pm 0.095$ \\

General Theme & $0.903 \pm 0.028^{*}$ &
$0.360 \pm 0.051$ &
$0.376 \pm 0.058$ \\



0

## Resultados por Matéria

In [48]:
df_pec_9 = df_all[df_all.materia == "PEC_6_2019"].copy()
df_pec_9 = df_pec_9[df_pec_9.tipo_texto == "texto_preprocessado"].reset_index(drop=True)

display(
    df_pec_9[["embedding", "precision@1", "r_precision", "map"]].style.format({
        "precision@1": "{:.3f}",
        "precision@5": "{:.3f}",
        "precision@10": "{:.3f}",
        "ndcg@10": "{:.3f}",
        "map": "{:.3f}",
    })
)

,embedding,precision@1,r_precision,map
0,embedding__Octen__Octen_Embedding_8B__texto_preprocessado,0.867,0.658753,0.713
1,embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado,0.856,0.653239,0.699
2,embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado,0.852,0.645340,0.695
3,embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado,0.826,0.639876,0.684
4,embedding__gemini__gemini_embedding_2__sts__texto_preprocessado,0.852,0.621883,0.671
5,embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto_preprocessado,0.856,0.590560,0.642
6,embedding__openai__text_embedding_3_large__texto_preprocessado,0.799,0.576045,0.626
7,embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado,0.788,0.578752,0.617
8,embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado,0.799,0.568352,0.617
9,embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto_preprocessado,0.795,0.507710,0.547


In [49]:
df_mpv_612 = df_all[df_all.materia == "MPV_612_2013"].copy()
df_mpv_612 = df_mpv_612[df_mpv_612.tipo_texto == "texto_preprocessado"].reset_index(drop=True)

display(
    df_mpv_612[["embedding", "precision@1", "r_precision", "map"]].style.format({
        "precision@1": "{:.3f}",
        "precision@5": "{:.3f}",
        "precision@10": "{:.3f}",
        "ndcg@10": "{:.3f}",
        "map": "{:.3f}",
    })
)

,embedding,precision@1,r_precision,map
0,embedding__Octen__Octen_Embedding_8B__texto_preprocessado,0.765,0.624949,0.670
1,embedding__gemini__gemini_embedding_2__sts__texto_preprocessado,0.782,0.627130,0.667
2,embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado,0.788,0.614590,0.663
3,embedding__joaorobson__KaLM_Embedding_Gemma3_12B_2511__texto_preprocessado,0.810,0.599140,0.648
4,embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado,0.765,0.600504,0.644
5,embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado,0.721,0.577354,0.624
6,embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado,0.777,0.576240,0.624
7,embedding__openai__text_embedding_3_large__texto_preprocessado,0.771,0.582640,0.621
8,retrieval__bm25l__texto_preprocessado__preprocess,0.782,0.560455,0.596
9,embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado,0.777,0.525949,0.570


In [50]:
df_plp_68 = df_all[(df_all.nivel == "tema") & (df_all.materia == "PLP_68_2024")].copy()
df_plp_68 = df_plp_68[df_plp_68.tipo_texto == "texto_preprocessado"].reset_index(drop=True)

display(
    df_plp_68[["embedding", "precision@1", "r_precision", "map", "ndcg_all"]].style.format({
        "precision@1": "{:.3f}",
        "precision@5": "{:.3f}",
        "precision@10": "{:.3f}",
        "ndcg_all": "{:.3f}",
        "map": "{:.3f}",
    })
)

,embedding,precision@1,r_precision,map,ndcg_all
0,embedding__gemini__gemini_embedding_2__sts__texto_preprocessado,0.837,0.588335,0.622,0.838
1,embedding__Octen__Octen_Embedding_8B__texto_preprocessado,0.818,0.570928,0.606,0.827
2,embedding__joaorobson__harrier_oss_v1_27b__texto_preprocessado,0.811,0.568953,0.604,0.830
3,embedding__codefuse_ai__F2LLM_v2_14B__texto_preprocessado,0.824,0.565604,0.598,0.826
4,embedding__Qwen__Qwen3_Embedding_8B__texto_preprocessado,0.816,0.554392,0.588,0.819
5,embedding__ICT_TIME_and_Querit__ICT_TIME_and_Querit_embedding_v1__texto_preprocessado,0.818,0.534817,0.561,0.812
6,retrieval__bm25l__texto_preprocessado__preprocess,0.793,0.528529,0.559,0.809
7,embedding__nvidia__llama_embed_nemotron_8b__texto_preprocessado,0.809,0.528156,0.557,0.803
8,embedding__jinaai__jina_embeddings_v5_text_small__text_matching__texto_preprocessado,0.809,0.529498,0.556,0.798
9,embedding__openai__text_embedding_3_large__texto_preprocessado,0.803,0.520260,0.549,0.806
